# 📊 Calories Burned 예측 — 탐색적 데이터 분석 (EDA)

## 프로젝트 개요
- **목표**: 운동 데이터 기반 칼로리 소모량 예측 (회귀 문제)
- **타겟 변수**: `Calories_Burned`
- **분석 구성**: 5단계 체계적 EDA

## 분석 단계
1. 데이터 품질 점검 (결측치, 중복, 이상치)
2. 타겟 변수 분포 분석 (편향, 정규성 검정)
3. 수치형 변수-타겟 관계 (상관계수, 산점도)
4. 범주형 변수 분석 (Gender, Weight_Status)
5. 피처 엔지니어링 탐색 (BMI, Interaction)

## 주요 발견
- 타겟 우측 편향 (skew ~0.82) → 로그 변환 필요
- 핵심 변수: Exercise_Duration(0.955), BPM(0.900), Body_Temperature(0.824)
- Weight, Height는 단독 설명력 거의 없음
- 성별 차이 존재하나 크지 않음

In [ ]:
# 필요한 라이브러리 임포트
import pandas as pd
import random
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import optuna
import warnings
warnings.filterwarnings('ignore')

from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.metrics import make_scorer, mean_squared_error
from scipy import stats

print("라이브러리 로딩 완료!")

In [ ]:
# 한글 폰트 설정 (깨지면 코렙파일에서 가져오기)


# 시각화 스타일 설정
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)



In [ ]:
# Fixed RandomSeed 항상 동일한 결과값 나오게 고정
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42) # Seed 고정

In [ ]:
# 데이터 로드
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f"데이터 형태: {train.shape}")
print(f"\n컬럼 정보:")
train.info()

In [ ]:
# 데이터 미리보기
train.head(10)

---
## 1단계: 데이터 품질 점검

결측치, 중복행, 이상치 확인,Height 통합

In [ ]:
# 1-1. 결측치 확인
print("=== 결측치 현황 ===")
missing = train.isnull().sum()
missing_pct = 100 * train.isnull().sum() / len(train)
missing_table = pd.DataFrame({'결측치 개수': missing, '비율(%)': missing_pct})
print(missing_table[missing_table['결측치 개수'] > 0])

if missing.sum() == 0:
    print("\n✓ 결측치 없음")

In [ ]:
# 1-2. 중복 행 확인
print("=== 중복 행 현황 ===")
duplicates = train.duplicated().sum()
print(f"중복 행 개수: {duplicates}")

if duplicates == 0:
    print("✓ 중복 행 없음")

In [ ]:
# 1-3. Height 통합 (Feet + Inches)
train['Height_Total_Inches'] = train['Height(Feet)'] * 12 + train['Height(Remainder_Inches)']

print("=== Height 통합 완료 ===")
print(f"Height_Total_Inches 생성: {train['Height_Total_Inches'].describe()}")

In [ ]:
# 1-4. 수치형 변수 기초 통계
print("=== 수치형 변수 기초 통계량 ===")
train.describe()

In [ ]:
# 1-5. 이상치 탐지 - 체온, BPM 확인
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 체온
axes[0].boxplot(train['Body_Temperature(F)'].dropna())
axes[0].set_title('Body Temperature (F) - Boxplot', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Temperature (F)')
axes[0].grid(True, alpha=0.3)

# BPM
axes[1].boxplot(train['BPM'].dropna())
axes[1].set_title('BPM - Boxplot', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Beats Per Minute')
axes[1].grid(True, alpha=0.3)

# Exercise Duration
axes[2].boxplot(train['Exercise_Duration'].dropna())
axes[2].set_title('Exercise Duration - Boxplot', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Minutes')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 이상치 통계
print("\n=== 이상치 탐지 (IQR 방식) ===")
for col in ['Body_Temperature(F)', 'BPM', 'Exercise_Duration']:
    Q1 = train[col].quantile(0.25)
    Q3 = train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = train[(train[col] < lower) | (train[col] > upper)]
    print(f"{col}: {len(outliers)}개 이상치 ({len(outliers)/len(train)*100:.2f}%)")

Body Temperature: 중앙값 약 104.4°F (운동 데이터라 104°F는 정상 범위, 99°F는 “운동 거의 안 한 경우” 가능)
제거하면 위험함

BPM : 중앙값 95 운동시 120 가능 자연스러운 분포로 보임

Exercise Duration : 완전 정상 분포

In [ ]:
# 1-6. 비현실적인 값 확인
print("=== 비현실적인 값 확인 ===")

# BPM = 0 확인
zero_bpm = train[train['BPM'] == 0]
print(f"BPM = 0인 행: {len(zero_bpm)}개")

# 체온이 비현실적인 경우 (95F 미만 또는 110F 초과)
abnormal_temp = train[(train['Body_Temperature(F)'] < 95) | (train['Body_Temperature(F)'] > 110)]
print(f"비정상 체온 (< 95F 또는 > 110F): {len(abnormal_temp)}개")

# 운동 시간이 0 이하
zero_exercise = train[train['Exercise_Duration'] <= 0]
print(f"운동 시간 <= 0: {len(zero_exercise)}개")

---
## 2단계: 타겟 변수 분석

Calories_Burned의 분포를 확인하고 정규성 검정을 수행합니다.

In [ ]:
# 2-1. Calories_Burned 분포
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Histogram
axes[0, 0].hist(train['Calories_Burned'], bins=50, edgecolor='black', alpha=0.7, color='teal')
axes[0, 0].set_title('Calories Burned - Histogram', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Calories Burned')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(train['Calories_Burned'].mean(), color='red', linestyle='--', label=f'Mean: {train["Calories_Burned"].mean():.2f}')
axes[0, 0].axvline(train['Calories_Burned'].median(), color='green', linestyle='--', label=f'Median: {train["Calories_Burned"].median():.2f}')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# KDE Plot
train['Calories_Burned'].plot(kind='kde', ax=axes[0, 1], color='teal', linewidth=2)
axes[0, 1].set_title('Calories Burned - KDE', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Calories Burned')
axes[0, 1].set_ylabel('Density')
axes[0, 1].grid(True, alpha=0.3)

# Q-Q Plot
stats.probplot(train['Calories_Burned'], dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Calories Burned - Q-Q Plot', fontsize=14, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Boxplot
axes[1, 1].boxplot(train['Calories_Burned'], vert=True)
axes[1, 1].set_title('Calories Burned - Boxplot', fontsize=14, fontweight='bold')
axes[1, 1].set_ylabel('Calories Burned')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

히스토그램 : 오른쪽 편향(right-skewed) 분포, 평균(89.37) > 중앙값(77.00)오른쪽 꼬리(high values)가 평균을 끌어올린다

KDE: 약 50-60 칼로리 부근에서 최대 밀도, 분포 비대칭(우측 편향) - 대부분 짧은 운동 또는 저강도 운동 수행

Q-Q Plot : 왼쪽 - 실제 분포의 낮은 값들이 정규분포보다 더 많음, 

중간: 정규분포와 유사, 

오른쪽: 대부분은 정규분포와 비슷하거나 약간 낮음,극소수의 극단값(약 250-300 칼로리)존재함


In [ ]:
# 2-2. 타겟 변수 통계량
print("=== Calories_Burned 기초 통계량 ===")
print(f"평균: {train['Calories_Burned'].mean():.2f}")
print(f"중앙값: {train['Calories_Burned'].median():.2f}")
print(f"표준편차: {train['Calories_Burned'].std():.2f}")
print(f"왜도(Skewness): {train['Calories_Burned'].skew():.4f}")
print(f"첨도(Kurtosis): {train['Calories_Burned'].kurtosis():.4f}")
print(f"최솟값: {train['Calories_Burned'].min():.2f}")
print(f"최댓값: {train['Calories_Burned'].max():.2f}")

In [ ]:
# 2-3. 정규성 검정 (Shapiro-Wilk Test)
# 샘플 크기가 크면 샘플링하여 검정
if len(train) > 5000:
    sample = train['Calories_Burned'].sample(5000, random_state=42)
else:
    sample = train['Calories_Burned']

statistic, p_value = stats.shapiro(sample)
print("\n=== 정규성 검정 (Shapiro-Wilk Test) ===")
print(f"검정 통계량: {statistic:.6f}")
print(f"p-value: {p_value:.6f}")

if p_value < 0.05:
    print("결론: 정규분포를 따르지 않음 (p < 0.05)")
    print("→ 로그 변환 또는 다른 변환을 고려할 수 있습니다.")
else:
    print("결론: 정규분포를 따름 (p >= 0.05)")

---
## 3단계: 수치형 변수와 타겟의 관계

산점도와 상관계수를 통해 각 변수와 칼로리의 관계를 파악합니다.

In [ ]:
# 3-1. 주요 수치형 변수와 타겟의 Scatter Plot
numerical_cols = ['Exercise_Duration', 'Body_Temperature(F)', 'BPM',
                    'Height_Total_Inches', 'Weight(lb)', 'Age']

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, col in enumerate(numerical_cols):
    axes[idx].scatter(train[col], train['Calories_Burned'], alpha=0.5, s=20, color='teal')
    axes[idx].set_xlabel(col, fontsize=11)
    axes[idx].set_ylabel('Calories_Burned', fontsize=11)
    axes[idx].set_title(f'{col} vs Calories_Burned', fontsize=12, fontweight='bold')
    
    # 상관계수 표시
    corr = train[col].corr(train['Calories_Burned'])
    axes[idx].text(0.05, 0.95, f'Corr: {corr:.3f}', 
                    transform=axes[idx].transAxes, 
                    fontsize=10, verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

위 3개 (중요) : Exercise_Duration (0.955) , BPM (0.900) , Body_Temperature (0.824)

아래 3개 (영향 적음) : Height (0.02) , Weight (0.04) , Age (0.16)

In [ ]:
# 3-2. 상관계수 히트맵
corr_cols = numerical_cols + ['Calories_Burned']
correlation_matrix = train[corr_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.3f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Heatmap - Numerical Variables', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# 타겟과의 상관계수 정렬
print("\n=== Calories_Burned와의 상관계수 (내림차순) ===")
target_corr = correlation_matrix['Calories_Burned'].sort_values(ascending=False)
print(target_corr)

In [ ]:
# 3-3. 주요 변수 간 Pair Plot (샘플링)
# 데이터가 크면 샘플링
if len(train) > 1000:
    sample_df = train.sample(1000, random_state=42)
else:
    sample_df = train

key_cols = ['Exercise_Duration', 'Body_Temperature(F)', 'BPM', 'Calories_Burned']
sns.pairplot(sample_df[key_cols], diag_kind='kde', plot_kws={'alpha':0.6, 's':20})
plt.suptitle('Pair Plot - Key Variables', y=1.02, fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

Temp vs Calories 곡선.. 비션형관계

Body Temperature vs Duration도 곡선 형태

단순 선형 회귀 문제가 아니라 Duration 중심의 비선형 구조인것같다

Duration 중심 정밀 모델링 해야하고.. 비선형 처리 잘 해야겠음 (파생변수 제곱항, 구간화,트리모델이면 깊이조절)


---
## 4단계: 범주형 변수 분석

Gender와 Weight_Status별 칼로리 분포를 비교합니다.

In [ ]:
# 4-1. 범주형 변수 분포 확인
print("=== Gender 분포 ===")
print(train['Gender'].value_counts())
print(f"\n비율:\n{train['Gender'].value_counts(normalize=True) * 100}")

print("\n=== Weight_Status 분포 ===")
print(train['Weight_Status'].value_counts())
print(f"\n비율:\n{train['Weight_Status'].value_counts(normalize=True) * 100}")

In [ ]:
# 4-2. Gender별 Calories_Burned 분포
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Boxplot
train.boxplot(column='Calories_Burned', by='Gender', ax=axes[0])
axes[0].set_title('Calories Burned by Gender - Boxplot', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Calories Burned')
axes[0].get_figure().suptitle('')  # 기본 제목 제거

# Violin plot
sns.violinplot(data=train, x='Gender', y='Calories_Burned', ax=axes[1], palette='Set2')
axes[1].set_title('Calories Burned by Gender - Violin Plot', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Calories Burned')

# Histogram
for gender in train['Gender'].unique():
    subset = train[train['Gender'] == gender]['Calories_Burned']
    axes[2].hist(subset, bins=30, alpha=0.6, label=f'Gender {gender}', edgecolor='black')
axes[2].set_title('Calories Burned by Gender - Histogram', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Calories Burned')
axes[2].set_ylabel('Frequency')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 통계 요약
print("\n=== Gender별 Calories_Burned 통계량 ===")
print(train.groupby('Gender')['Calories_Burned'].describe())

성별 불균형없음 남성(M)이 여성(F)보다 중앙값이 약간 높음 칼로리소모량 성별차이 약간있을수있으나 중요하진않을듯
바이올린플롯 보면 비슷한데 남자가 우측꼬리 더 길어서 고칼로리구간 약간 더 많음
히스토그램도 거의 겹쳐서 영량은 있겠지만 큰 영향은 아닐듯
몸무게는 아까 히트맵에서 봤듯이 설명력 거의 없으니까 패스

In [ ]:
# 4-3. Weight_Status별 Calories_Burned 분포
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Boxplot
train.boxplot(column='Calories_Burned', by='Weight_Status', ax=axes[0])
axes[0].set_title('Calories Burned by Weight Status - Boxplot', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Weight Status')
axes[0].set_ylabel('Calories Burned')
axes[0].get_figure().suptitle('')
axes[0].tick_params(axis='x', rotation=45)

# Violin plot
sns.violinplot(data=train, x='Weight_Status', y='Calories_Burned', ax=axes[1], palette='Set3')
axes[1].set_title('Calories Burned by Weight Status - Violin Plot', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Weight Status')
axes[1].set_ylabel('Calories Burned')
axes[1].tick_params(axis='x', rotation=45)

# Histogram
for status in train['Weight_Status'].unique():
    subset = train[train['Weight_Status'] == status]['Calories_Burned']
    axes[2].hist(subset, bins=30, alpha=0.6, label=status, edgecolor='black')
axes[2].set_title('Calories Burned by Weight Status - Histogram', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Calories Burned')
axes[2].set_ylabel('Frequency')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 통계 요약
print("\n=== Weight_Status별 Calories_Burned 통계량 ===")
print(train.groupby('Weight_Status')['Calories_Burned'].describe())

세 그룹 다 중앙값 거의 비슷

바이올린플룻 분포 형태도 거의 동일 꼬리도 비슷 히스토그램도 분포 겹침 weight는 의미없는 변수로 확정

In [ ]:
# 4-4. Gender × Weight_Status 교차 분석
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Grouped Boxplot
sns.boxplot(data=train, x='Weight_Status', y='Calories_Burned', hue='Gender', ax=axes[0], palette='Set2')
axes[0].set_title('Calories Burned by Weight Status and Gender', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Weight Status')
axes[0].set_ylabel('Calories Burned')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(title='Gender')

# Average Calories by Group
avg_by_group = train.groupby(['Weight_Status', 'Gender'])['Calories_Burned'].mean().unstack()
avg_by_group.plot(kind='bar', ax=axes[1], width=0.8, color=['salmon', 'skyblue'])
axes[1].set_title('Average Calories Burned by Weight Status and Gender', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Weight Status')
axes[1].set_ylabel('Average Calories Burned')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Gender')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n=== Gender × Weight_Status별 평균 Calories_Burned ===")
print(train.groupby(['Gender', 'Weight_Status'])['Calories_Burned'].mean().unstack())

성별 차이는 존재하지만, 매우 크지는 않다. 몸무게 구간별 성별차이가 동일하지는 않음 (Gender × Weight_Status 상호작용이 강하지 않음)

---
## 5단계: 피처 엔지니어링 탐색

BMI 계산 및 변수 간 interaction 탐색

In [ ]:
# 5-1. BMI 계산
# BMI = weight(lb) / (height(inches))^2 * 703
train['BMI'] = (train['Weight(lb)'] / (train['Height_Total_Inches'] ** 2)) * 703

print("=== BMI 통계량 ===")
print(train['BMI'].describe())

# BMI 카테고리 생성
def bmi_category(bmi): 
    if bmi < 18.5:
        return 'Underweight'
    elif 18.5 <= bmi < 25:
        return 'Normal Weight'
    elif 25 <= bmi < 30:
        return 'Overweight'
    else:
        return 'Obese'

train['BMI_Category'] = train['BMI'].apply(bmi_category)

print("\n=== BMI 카테고리 분포 ===")
print(train['BMI_Category'].value_counts())

In [ ]:
# 5-2. BMI와 Calories_Burned의 관계
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot
axes[0].scatter(train['BMI'], train['Calories_Burned'], alpha=0.5, s=20, color='teal')
axes[0].set_xlabel('BMI', fontsize=12)
axes[0].set_ylabel('Calories Burned', fontsize=12)
axes[0].set_title('BMI vs Calories Burned', fontsize=14, fontweight='bold')
corr_bmi = train['BMI'].corr(train['Calories_Burned'])
axes[0].text(0.05, 0.95, f'Correlation: {corr_bmi:.3f}', 
            transform=axes[0].transAxes, fontsize=11, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
axes[0].grid(True, alpha=0.3)

# Boxplot by BMI Category
train.boxplot(column='Calories_Burned', by='BMI_Category', ax=axes[1])
axes[1].set_title('Calories Burned by BMI Category', fontsize=14, fontweight='bold')
axes[1].set_xlabel('BMI Category')
axes[1].set_ylabel('Calories Burned')
axes[1].get_figure().suptitle('')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\n=== BMI 카테고리별 평균 Calories_Burned ===")
print(train.groupby('BMI_Category')['Calories_Burned'].mean().sort_values(ascending=False))

In [ ]:
# 5-3. Interaction Features 탐색
# 운동 시간 × BPM
train['Duration_x_BPM'] = train['Exercise_Duration'] * train['BPM']

# 운동 시간 × 체온
train['Duration_x_Temp'] = train['Exercise_Duration'] * train['Body_Temperature(F)']

# BPM × 체온
train['BPM_x_Temp'] = train['BPM'] * train['Body_Temperature(F)']

# Interaction features와 타겟의 상관계수
interaction_cols = ['Duration_x_BPM', 'Duration_x_Temp', 'BPM_x_Temp']

print("=== Interaction Features와 Calories_Burned 상관계수 ===")
for col in interaction_cols:
    corr = train[col].corr(train['Calories_Burned'])
    print(f"{col}: {corr:.4f}")

In [ ]:
# 5-4. Interaction Features 시각화
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, col in enumerate(interaction_cols):
    axes[idx].scatter(train[col], train['Calories_Burned'], alpha=0.5, s=20, color='teal')
    axes[idx].set_xlabel(col, fontsize=11)
    axes[idx].set_ylabel('Calories_Burned', fontsize=11)
    axes[idx].set_title(f'{col} vs Calories_Burned', fontsize=12, fontweight='bold')
    
    corr = train[col].corr(train['Calories_Burned'])
    axes[idx].text(0.05, 0.95, f'Corr: {corr:.3f}', 
                    transform=axes[idx].transAxes, fontsize=10, verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 5-5. 최종 상관계수 비교 (원본 vs 파생변수)
all_features = numerical_cols + ['BMI'] + interaction_cols
final_corr = train[all_features + ['Calories_Burned']].corr()['Calories_Burned'].sort_values(ascending=False)

print("\n=== 모든 Features와 Calories_Burned 상관계수 (내림차순) ===")
print(final_corr)

# 시각화
plt.figure(figsize=(10, 8))
final_corr[:-1].sort_values().plot(kind='barh', color='teal')
plt.title('Feature Importance - Correlation with Calories_Burned', fontsize=14, fontweight='bold')
plt.xlabel('Correlation Coefficient')
plt.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

---
##  EDA 요약 및 결론

In [ ]:
print("="*80)
print("EDA 주요 발견사항")
print("="*80)

print("\n 1.데이터 품질")
print(f"   - 총 데이터: {len(train):,}행")
print(f"   - 결측치: {train.isnull().sum().sum()}개")
print(f"   - 중복행: {train.duplicated().sum()}개")
print("   - Height를 Total_Inches로 통합 완료")

print("\n 2. 타겟 변수 (Calories_Burned)")
print(f"   - 평균: {train['Calories_Burned'].mean():.2f}")
print(f"   - 왜도: {train['Calories_Burned'].skew():.4f} ({'오른쪽 편향' if train['Calories_Burned'].skew() > 0 else '왼쪽 편향'})")
print("   - 정규성 검정 결과에 따라 변환 필요 여부 결정")

print("\n 3. 주요 상관관계 (Top 3)")
top_corr = final_corr[final_corr.index != 'Calories_Burned'].head(3)
for i, (feature, corr) in enumerate(top_corr.items(), 1):
    print(f"   {i}. {feature}: {corr:.4f}")

print("\n 4. 범주형 변수 인사이트")
print(f"   - Gender별 평균 칼로리: ")
for gender in train['Gender'].unique():
    avg = train[train['Gender'] == gender]['Calories_Burned'].mean()
    print(f"     {gender}: {avg:.2f}")

print("\n 5. 피처 엔지니어링")
print("   - BMI 생성: Weight_Status보다 연속형 변수로 유용할 가능성")
print("   - Interaction Features:")
for col in interaction_cols:
    corr = train[col].corr(train['Calories_Burned'])
    print(f"     {col}: {corr:.4f}")

print("\n" + "="*80)
print(" EDA 완료 - 모델링 준비 완료")
print("="*80)

# 파생변수 생성

In [ ]:
# 최종 데이터셋 미리보기 (파생변수 포함)
print("\n=== 최종 데이터셋 (파생변수 포함) ===")
display_cols = ['Exercise_Duration', 'Body_Temperature(F)', 'BPM', 
                'Height_Total_Inches', 'Weight(lb)', 'Gender', 'Weight_Status',
                'BMI', 'BMI_Category', 'Calories_Burned']
train[display_cols].head(10)